##### In this guide, I’ll walk you through everything you need to know to get started with Databricks, a powerful platform for data engineering, data science, and machine learning. From setting up your environment to understanding key features like data processing and orchestration, this guide has got you covered. Let’s jump right in!

### DataFrame

A DataFrame is a distributed collection of data organized into named columns. It is conceptually equivalent to a table in a relational database or a data frame in R/Python but with richer optimizations under the hood. DataFrames can be constructed from a wide array of sources such as structured data files, tables in Hive, external databases, or existing RDDs.

###Lazy Evaluation

Spark’s lazy evaluation comes into play at this point. The logical execution plan is not immediately executed, and Spark defers the computation until an action is called.

### Transformations

Transformations are the instructions you use to modify the DataFrame in the way you want and are lazily executed. There are two types of transformations: narrow and wide.

**Narrow Transformations:** These are transformations for which each input partition will contribute to only one output partition. Examples include map, filter, and select.

**Wide Transformations:** These transformations will have input partitions contributing to many output partitions. Spark will perform a data shuffle operation, redistributing partitions across the cluster. Examples include group by and join.

### Actions

Actions are operations that trigger the data processing and return results or write data to storage. Examples of actions include count, collect, write, show, etc. When you call an action, Spark evaluates the entire logical execution plan built through transformations and optimizes the execution plan before executing it.



### Cluster Types

##### All Purpose Cluster

**All Purpose Cluster:** These clusters are primarily used for interactive data analysis using Databricks notebooks. Multiple users can share these clusters for collaborative interactive analysis. Clusters without any activity are terminated after the specified time in the “terminate” field of the cluster configuration.

**Job Cluster:** Job Clusters in Databricks are transient clusters specifically created for running jobs. Running non-interactive workloads is preferred for cost efficiency, as the compute is only used for the duration of the job, after which it will be terminated.

**Serverless SQL Warehouse:** This type of cluster is for executing SQL queries against tables. Access is provided through a SQL editor, and there are predefined clusters available for configuration.

**Python Serverless:** This type of cluster allows executing PySpark or SQL code. It starts in just a few seconds.

In [0]:
data = [[1, "VW"],  
        [2, "BMW"]] 

columns = ["ID", "Car"] 

dataframe = spark.createDataFrame(data, columns) 
  
# show data frame 
display(dataframe)

####Reading and Saving Data from Various Sources
Spark enables us to effortlessly read and save data from various sources, including CSV files, Parquet, Avro, JSON, and more.

The read method in Spark allows us to import data from other databases such as Oracle, PostgreSQL, and more.

In [0]:
#CSV
df = (
                spark.read
                .option("delimiter", ",")
                .option("header", True)
                .csv(pat_to_file)
            )

In [0]:
#SQL
SELECT * FROM read_files(
  '/Volumes/demo/bronze/landing/orders.csv',
  format => 'csv',
  header => true,
  mode => 'FAILFAST')

-- OR

SELECT * 
FROM 
csv.`/mnt/path/to/file`

####### While Spark can automatically detect schema for us, there is also an option to declare it manually.

In [0]:
from pyspark.sql.types import *

cust_schema = StructType([
    StructField("id", IntegerType()),
    StructField("car", StringType()),
])

df = (
                spark.read
                .option("delimiter", ",")
                .option("header", True)
                .option("cust_schema")
                .csv(pat_to_file)
            )

In [0]:
#Oracle
query = "select 1 a from dueal"

df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:oracle:thin:user/pass@//address:port/instance") \
    .option("query", query) \
    .option("driver", "oracle.jdbc.driver.OracleDriver") \
    .load()

##### Saving data

We can save files directly to storage using the following code:



In [0]:
df.write
.format("csv")
.option("header", "true")
.save(path_to_save)

######
Databricks tables

When we have data loaded in our data frame, we can transform them using transformations, save the data frame on storage, or create a table in a schema(database). To save data in a table, we need to create a database, which we can do using Spark SQL:

In [0]:
%sql
CREATE SCHEMA test_db

In [0]:
dataframe.write.mode("overwrite").saveAsTable("test_db.cars")

In [0]:
%sql
-- Creates a Delta table

CREATE TABLE test_db.cars(
id int,
model STRING,
year int
)

-- Use data from another table
CREATE TABLE test_db.cars 
AS 
SELECT * from load_cars


-- Use data from another table using its path
CREATE TABLE IF NOT EXISTS test_db.cars
AS SELECT * 
FROM 
delta.`/Volumes/demo/raw/cars`;

### External Table
If we specify a location, it will result in the creation of a Spark unmanaged table(External Table).

The main difference between them is that Databricks only manages the metadata; when you drop a table, you do not remove the underlying data

In [0]:
%sql
CREATE TABLE test_db.cars(
id int,
model STRING,
year int
)
USING DELTA
LOCATION '/Volumes/demo/silver/cars';

#### Read Databricks tables

With Spark and SQL, we can execute a query against a table existing in Databricks meta store to retrieve data from them.

In [0]:
df = spark.sql("""select * from test_db.cars""")

In [0]:
display(df)

In [0]:
df = spark.read.table("test_db.cars")


In [0]:
display(df)

Databricks notebooks allow us to query tables using Spark SQL directly. To use SQL, we need to switch the language to SQL or use the magic command %sql. The result will be displayed under the cell with the SQL command.

In [0]:
%sql

select * from test_db.cars

## Delta Lake

Delta Lake is the default storage format for all operations on Databricks. Unless otherwise specified, all tables on Databricks are Delta tables. Delta Lake serves as the optimized storage layer that forms the foundation for storing data and tables in the Databricks lakehouse. It is an open-source technology that extends Parquet data files with a file-based transaction log for ACID transactions and scalable metadata handling. Delta Lake seamlessly integrates with Apache Spark, offering a range of important features:

**Additional Operations:** Delta Lake provides additional operations such as inserts, updates, and deletes. You can update data in a target using the SQL Merge operation, improving load strategies for a lakehouse and supporting operations like Slowly Changing Dimensions (SCD).

**ACID Transactions:*** Delta Lake ensures ACID (Atomicity, Consistency, Isolation, Durability) transactions, guaranteeing data integrity and reliability.
Time Travel: Delta Lake offers time travel capabilities by providing a transaction log, enabling users to access and revert to earlier versions of data, facilitating data auditing and versioning.

**Schema Evolution:** Delta Lake supports schema evolution, allowing changes to a table schema that can be applied automatically without the need for recreating a file, ensuring flexibility and ease of maintenance.

**Optimization:** Delta Lake provides optimization features such as OPTIMIZE and ZORDER BY to reduce the number of files and physically organize data within a file, improving query performance and efficiency.

**Cache in Databricks:** Delta Lake allows caching files on worker nodes and caching the result of queries in memory, reducing the need for reading from files and enhancing query performance.

In [0]:
data = [
[1,"Alicia Kaiser","WA","001-677-774-4370","13714","Stacy Summit","54799","Lake Joseborough","Delaware","1939-04-07","2656508165701512", "1000.00"],
[2,"Donna Ellis","BR","001-631-995-8008","43599","Adam Trail","07204","Port April","Montana","1966-07-28","5527331190171381",  "1000.00"],
[3,"Kenneth Smith","WA","001-592-849-6009x4173","649","Sherri Grove","14527","North Miranda","Washington","1998-09-25","5366314062583069",  "1000.00"],
[4,"Danny Clark","WA","574.419.0221x348","285","Timothy Drive","41106","West Erica","Maryland","1948-08-29","5488489084734990",  "2000.00"],
[5,"Nicholas Thompson","CA","(259)268-8760x061","998","Russell Shoals","65647","South Todd","South Carolina","1985-03-08","2720282150775392", "3000.00"],
[6,"Frances Griffith","WA","7535316823","9559","Emily Branch","71422","Mcdanielhaven","New York","1984-07-12","2248334835679706", "4000.00"],
[7,"Trevor Harrington","CA","742.224.9375","5960","Lisa Port","73881","Loganbury","New York","1979-03-10","5130498353342015", "2000.00"],
[8,"Seth Mitchell","AA","(386)517-7589x04440","47352","Stafford Loop","01347","South Alexander","North Dakota","1944-07-03","2280935548220544", "3000.00"],
[9,"Patrick Caldwell","BR","001-307-225-9094","0170","Amanda Dam","24885","Port Mollyhaven","Connecticut","1973-05-04","5557276831443314", "2000.00"],
[10,"Laura Hopkins","CA","9095819755","143","Lee Brook","23623","Jarvisland","Hawaii","1971-11-13","2720224762678291", "100.00"]
]
schama = "client_number int,name string,branch string,phone_number string,bulding_number string,street_name string,postcode string,city string,state string,birth_date string,credit_card_number string, amout string"
df = spark.createDataFrame(data, schama)

In [0]:
display(df)

In [0]:
#selecing columns
df.select("client_number","name","amout")

Spark allows us to use SQL to transform data. To utilize a DataFrame created or imported within a SQL context, we need to create a temporary view:

In [0]:
df.createTempView("client")

In [0]:
%sql
select client_number,name,amout from client

There are various options to access columns in a DataFrame. Below, you can find a few examples:

In [0]:
from pyspark.sql.functions import col,expr,column
display(df.select(expr("client_number as client_num"),col("client_number").alias("client_num")).limit(3))

During data transformation, common scenarios include changing data types, renaming columns, adding new columns, and deriving new columns based on values from others. Let’s explore how we can achieve these tasks.

In [0]:
from pyspark.sql.functions import expr, col, column, lit, exp, current_date

age_exp = "extract( year from current_date) - extract( year from birth_date) "

df1 = df.withColumn("amout", col("amout").cast("decimal(10,2)")) \
     .withColumn("birth_date", col("birth_date").cast("date")) \
     .withColumn("age", expr(age_exp)) \
     .withColumn("load_date", lit(current_date()))

df1 = df1.drop("credit_card_number", "birth_date")

display(df1)

### Filtering Rows in a DataFrame

Filtering rows in a DataFrame involves creating a condition that separates the data you want to keep from the data you don’t. This condition can be written as a simple expression or built from multiple comparisons. DataFrames offer two methods, where and filter, to achieve this filtering based on your chosen condition.

In [0]:
display(df1.where("age >= 85"))
display(df1.where(col("age") >= 85))

It’s possible to build more complex expressions in DataFrame filtering using AND or OR conditions, allowing for greater flexibility in specifying conditions for row selection.

In [0]:
display(df1.where("age >= 85 or age <=30"))
display(df1.filter( (col("age") >= 85) |  (col("age") <= 30)))

In [0]:
display(df1.where("age >=30 and branch != 'WA' "))
display(df1.filter( (col("age") >= 30) &  (col("branch") != 'WA')))

### Grouping and Sorting

Group by is a transformation operation in PySpark used to group the data in a Spark DataFrame based on specified columns. This operation is often followed by aggregating functions such as count(), sum(), avg(), etc., allowing for the summarization of grouped data.

Order by, on the other hand, sorts records in a DataFrame based on specified sort conditions, allowing for the arrangement of data in ascending or descending order.

In [0]:
from pyspark.sql.functions import desc, count, sum

# count by brnahc
display(df1.groupBy("branch").count().orderBy(desc("count")))

# sum, count by branch with order by client_number count
display(df1.groupBy("branch").agg( 
sum("amout").alias("amout"), count("client_number").alias("qt") 
).orderBy(desc("qt")))

### Joining DataFrames

Joining operations are crucial for various data processing tasks such as data normalization, data modeling, and ensuring data quality. Spark supports joins using DataFrame join and SQL joins.

To demonstrate the join operation, we need an additional DataFrame. I’ll create it similarly so you can easily replicate my steps, or you can load data from a file or table.

In [0]:
import uuid

trans = [
    [str(uuid.uuid4()), 1, 100.00, '2024-02-01'],
    [str(uuid.uuid4()), 1, 200.00, '2024-02-03'],
    [str(uuid.uuid4()), 1, 130.00, '2024-02-04'],
    [str(uuid.uuid4()), 2, 110.00, '2024-02-05'],
    [str(uuid.uuid4()), 3, 200.00, '2024-02-01'],
    [str(uuid.uuid4()), 2, 300.00, '2024-02-02'],
    [str(uuid.uuid4()), 2, 50.00, '2024-02-03'],

]

schama = "id string, client_id int, value double, tran_date string"
df_tran = spark.createDataFrame(trans, schama)
display(df_tran)

The code below illustrates how to join two DataFrames. Spark supports various join types, including:

Inner Join

Left / Left Outer Join

Right / Right Outer Join

Outer / Full Join

Cross Join

Left Anti Join

Left Semi Join

In [0]:
import uuid

trans = [
    [str(uuid.uuid4()), 1, 100.00, '2024-02-01'],
    [str(uuid.uuid4()), 1, 200.00, '2024-02-03'],
    [str(uuid.uuid4()), 1, 130.00, '2024-02-04'],
    [str(uuid.uuid4()), 2, 110.00, '2024-02-05'],
    [str(uuid.uuid4()), 3, 200.00, '2024-02-01'],
    [str(uuid.uuid4()), 2, 300.00, '2024-02-02'],
    [str(uuid.uuid4()), 2, 50.00, '2024-02-03'],

]

schama = "id string, client_id int, value double, tran_date string"
df_tran = spark.createDataFrame(trans, schama)
display(df_tran)

In [0]:
# column name index style
display(df.join(df_tran, df['client_number'] == df_tran['client_id'], 'inner'))

# column name property style
display(df.join(df_tran, df.client_number == df_tran.client_id, 'inner'))

- Left Anti join: Retrieves records from the left DataFrame that do not exist in the right DataFrame.

In [0]:
display(df.join(df_tran, df['client_number'] == df_tran['client_id'], 'left_anti').select("client_number"))


Left Semi-join: Retrieves records and columns from the left DataFrame where records match records in the right DataFrame.

In [0]:
display(df.join(df_tran, df['client_number'] == df_tran['client_id'], 'leftsemi'))


### Union DataFrames

Spark facilitates union operations on DataFrames in several ways. We can union DataFrames using column names, as shown in example 1. Alternatively, we can union DataFrames without checking column names, as demonstrated in example 2. Additionally, Spark allows for merging two DataFrames while allowing missing columns.

In [0]:
df1 = spark.createDataFrame([[1, 2, 3]], ["col0", "col1", "col2"])

df2 = spark.createDataFrame([[4, 5, 6]], ["col1", "col2", "col0"])

# example 1
display(df1.unionByName(df2))

# example 2
display(df1.union(df2))

df2 = spark.createDataFrame([[4, 5, 6]], ["col1", "col2", "col3"])

# example 3
display(df1.unionByName(df2, allowMissingColumns=True))

###When function
A useful function in PySpark is when, which is employed in cases where we need to translate or map a value to another value based on specified conditions.

In [0]:
from pyspark.sql.functions import expr, col, column, lit, exp, current_date, when

data = [("Robert", "Smith","M",40000),("Ana","Novak","M",60000),
        ("Carl","Xyz",None,500000),("Maria","Free","F",500000),
        ("Johm","Az","",None), ("Steve","Smith","",1000)]

columns = ["name","surname","gender","salary"]
df = spark.createDataFrame(data = data, schema = columns)

df2 = df.withColumn("new_gender", when(df.gender == "M","Male")
                                 .when(df.gender == "F","Female")
                                 .when(df.gender.isNull() ,"")
                                 .otherwise(df.gender))

### Databricks Auto Loader
Auto Loader processes new data files incrementally and efficiently as they appear in storage. It provides a Structured Streaming source called cloudFiles. After specifying the input directory path, the cloudFiles source automatically processes new files as soon as they arrive, with the option to also process existing files in that directory. Auto Loader supports both Python and SQL in Lakeflow Declarative Pipelines.

Auto Loader can be configured in two modes:

**Listing Mode** — Lists all files in a directory and compares them with a RocksDB database to identify new files.

**Event Mode**— Reads file creation events and compares them with a RocksDB database to identify new files.

#### Python Functions and Modules
In Databricks, we can organize our code using either notebooks or Python modules. When using notebooks, the process is straightforward. We create a notebook, for example, named “Utils,” where we define all the functions that we commonly use in our solution. Then, we can call the %run command to include the defined functions in other notebooks.

In [0]:
%run fun

print(add(1,2))

In [0]:
# fun.py
def add(a,b):
    return a+b

In [0]:
#We can import this module using well-known Python syntax:
from utils.fun import add


print(add(1,2))

#### Unity Catalog
Databricks Unity Catalog is a unified governance solution that centralizes access control, auditing, lineage, and data discovery capabilities across Databricks workspaces. Key features of Unity Catalog include security, data discovery, lineage, and data sharing.

Security: Unity Catalog offers a single place to administer data access policies that apply across all workspaces.
Data Discovery: Unity Catalog lets you tag and document data assets and provides a search interface to help data consumers find data.
Data Lineage: Data lineage supports use cases such as tracking and monitoring jobs, debugging failures, understanding complex workflows, and tracing transformation rules. It presents the flow of data in user-friendly diagrams.
Data Sharing: Unity Catalog gives businesses more control over how, why, and what data is being shared with whom.

##### The Unity Catalog Object Model
The Unity Catalog object model uses a tree-level namespace to address various types of data assets in the catalog. The top-level meta store acts as a global container for assets such as catalogs, followed by schemas for all your data entities like tables, views, models, and functions.

##### Other elements:

**Volumes:** Provide access to non-tabular data stored in cloud object storage.

**Lakehouse Federation:** Provides direct access to relational database engines such as MySQL, PostgreSQL, Amazon Redshift, Snowflake, Microsoft SQL Server, Azure Synapse (SQL Data Warehouse), and Google BigQuery via Databricks.

#### Databricks Workflows
Databricks Workspace provides “Workflows” functionality supporting job orchestration and scheduling. If you prefer to work exclusively within the Databricks environment or with cloud providers such as AWS or GCP, this option is suitable. Workflows help create tasks and orchestrate steps in data processing processes. For detailed configuration instructions.